In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import torch

sys.path.append('.')
from src.utils import *
from src.visualization import *
from src.engine import *
from src.dataset import *

In [ ]:
set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 0  # Windows/Jupyter에서는 0이 가장 안정적
print(f"Device: {device} | num_workers: {NUM_WORKERS}")

run_dirs = make_run_dir('./outputs')

In [ ]:
from src.engine import run_repeated_holdout_a2

mobilenet_results = run_repeated_holdout_a2(
    model_name      = "mobilenet",
    bad_weight_list = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0],
    seeds           = [42, 123, 456],
    device          = device,
    n_splits        = 5,
    num_epochs_cv   = 40,
    num_epochs_final= 40,
    batch_size      = 16,
    num_workers     = NUM_WORKERS,
    early_stop_patience = 10,
    test_size       = 0.115,   # ~55장
    champion_select = "mean",
)

In [ ]:
import pandas as pd

df = pd.DataFrame(mobilenet_results)[["seed", "best_weight", "test_precision", "test_recall", "test_f2"]]
df.columns = ["Seed", "Best Weight", "Precision", "Recall", "F2"]
display(df)
print(f"\nF2  Mean ± Std : {df['F2'].mean():.4f} ± {df['F2'].std():.4f}")
print(f"Rec Mean ± Std : {df['Recall'].mean():.4f} ± {df['Recall'].std():.4f}")